In [4]:
import re
from pathlib import Path
import spacy

# =========================
# Setup
# =========================
nlp = spacy.load("en_core_web_sm")

NON_ACTIVITY_VERBS = {"ensure", "allow", "shall", "must", "require", "verify", "guarantee"}

_UNIT_CANON = {
    "cm": "cm", "centimeter": "cm", "centimeters": "cm",
    "s": "s", "sec": "s", "second": "s", "seconds": "s",
    "v": "v", "volt": "v", "volts": "v",
    "degree": "deg", "degrees": "deg",
}

RE_NUM_UNIT = re.compile(
    r"(?P<num>\d+(?:\.\d+)?)\s*(?P<unit>cm|centimeters?|seconds?|second|s|deg(?:rees)?|degrees?|v|V|volts?|volt)\b"
)

# =========================
# Helpers
# =========================
def _normalize_ws(s: str) -> str:
    return " ".join(s.split()).strip()


def _canon_unit(u: str) -> str:
    u0 = u.strip().lower()
    if u0.startswith("deg"):
        return "deg"
    return _UNIT_CANON.get(u0, u0)


def _constraint_type(op: str):
    # '=' -> None (blank) as requested
    return {
        "<=": "upper_bound",
        ">=": "lower_bound",
        "<": "strict_upper_bound",
        ">": "strict_lower_bound",
        "=": None,
    }.get(op, None)


def noun_phrase_text(token, drop_dets=True) -> str:
    toks = list(token.subtree)
    if drop_dets:
        toks = [t for t in toks if t.dep_ != "det"]
    return _normalize_ws(" ".join(t.text for t in toks))


def lemmatize_phrase(text: str) -> str:
    doc = nlp(text)
    out = []
    for t in doc:
        if t.is_space or t.is_punct:
            continue
        if t.pos_ == "DET":
            continue
        lem = (t.lemma_ or t.text).lower()
        if lem == "-pron-":
            lem = t.text.lower()
        out.append(lem)
    return _normalize_ws(" ".join(out))


def _add_unique(item, lst, seen):
    if item and item not in seen:
        seen.add(item)
        lst.append(item)


# =========================
# Load requirements (from file)
# =========================
def load_requirements(path: str = "requirements_input.txt") -> list[str]:
    """
    Reads requirements from a plain text file (1 requirement per line).
    Empty lines are ignored.
    """
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"Requirements file not found: {path}. "
            f"Make sure '{path}' exists in the repo next to main.py."
        )

    lines = p.read_text(encoding="utf-8").splitlines()
    reqs = [ln.strip() for ln in lines if ln.strip()]
    if not reqs:
        raise ValueError(f"No requirements found in {path}. Add at least one non-empty line.")
    return reqs


# =========================
# SYSTEM extraction
# =========================
def extract_systems(doc):
    systems, seen = [], set()
    for tok in doc:
        if tok.lemma_.lower() == "shall":
            best = None
            for chunk in doc.noun_chunks:
                if chunk.end <= tok.i and chunk.end > (best.end if best else -1):
                    best = chunk
            if best:
                name = re.sub(r"^\s*(the|a|an)\s+", "", best.text.strip(), flags=re.I)
                _add_unique(name, systems, seen)
            break
    return systems


# =========================
# ACTIVITY extraction
# =========================
def _is_modifier_verb(vtok):
    return vtok.dep_ in {"amod", "acl", "compound"} and vtok.head.pos_ in {"NOUN", "PROPN"}


def _collect_conj_verbs(vtok):
    verbs = [vtok]
    for ch in vtok.children:
        if ch.dep_ == "conj" and ch.pos_ == "VERB":
            verbs.append(ch)
    return verbs


def _verb_object_surface(vtok):
    # 1) direct object
    for ch in vtok.children:
        if ch.dep_ in {"dobj", "obj", "attr"}:
            return noun_phrase_text(ch).lower()

    # 2) clause complement (for "indicate when ... avoidance maneuver")
    for ch in vtok.children:
        if ch.dep_ in {"ccomp", "advcl"}:
            nouns = [t for t in ch.subtree if t.pos_ in {"NOUN", "PROPN"}]
            for n in nouns:
                if n.dep_ in {"dobj", "obj", "attr"}:
                    return noun_phrase_text(n).lower()
            if nouns:
                return noun_phrase_text(nouns[-1]).lower()

    return None


def _first_shall(doc):
    for t in doc:
        if t.lemma_.lower() == "shall":
            return t
    return None


def _pick_main_verb_after_shall(doc):
    shall_tok = _first_shall(doc)
    if not shall_tok:
        return None
    for t in doc[shall_tok.i + 1:]:
        if t.pos_ == "VERB":
            return t
    return None


def _unwrap_meta_to_complement(v):
    if not v:
        return v
    if (v.lemma_ or v.text).lower() in NON_ACTIVITY_VERBS:
        for ch in v.children:
            if ch.dep_ in {"xcomp", "ccomp"} and ch.pos_ == "VERB":
                return ch
    return v


def _prefer_to_infinitive_over_use(v):
    if not v:
        return v
    if (v.lemma_ or v.text).lower() == "use":
        for ch in v.children:
            if ch.dep_ == "xcomp" and ch.pos_ == "VERB":
                return ch
    return v


def extract_activities(doc):
    acts, seen = [], set()

    v0 = _pick_main_verb_after_shall(doc)
    if not v0:
        return acts

    v0 = _unwrap_meta_to_complement(v0)
    v0 = _prefer_to_infinitive_over_use(v0)

    for v in _collect_conj_verbs(v0):
        if not v or v.pos_ != "VERB":
            continue
        if _is_modifier_verb(v):
            continue

        vlem = (v.lemma_ or v.text).lower()
        if vlem in NON_ACTIVITY_VERBS:
            continue

        obj = _verb_object_surface(v)
        act = _normalize_ws(f"{vlem} {obj}") if obj else vlem
        _add_unique(act, acts, seen)

    return acts


# =========================
# ATTRIBUTES & CONSTRAINTS
# =========================
def _extract_via_method(doc):
    for tok in doc:
        if tok.text.lower() == "via" and tok.dep_ == "prep":
            for child in tok.children:
                if child.dep_ == "pobj":
                    span = doc[child.left_edge.i: child.right_edge.i + 1]
                    return _normalize_ws(span.text)
    return None


def extract_attributes_and_constraints(req):
    doc = nlp(req)
    text_low = req.lower()

    attributes_req, attr_seen = [], set()
    pairs = []

    def add_attr(attr_text):
        attr = lemmatize_phrase(attr_text)
        _add_unique(attr, attributes_req, attr_seen)
        return attr

    def add_pair(attr_name, op, val, unit):
        pairs.append((attr_name, {"type": _constraint_type(op), "value": val, "unit": unit}))

    # via USB or Bluetooth -> communication interface
    via_method = _extract_via_method(doc)
    if via_method:
        attr = add_attr("communication interface")
        add_pair(attr, "=", via_method, None)

    # attributes from noun chunks
    for chunk in doc.noun_chunks:
        c = chunk.text.lower()
        if any(k in c for k in ["distance", "threshold", "voltage", "battery state", "avoidance threshold"]):
            add_attr(chunk.text)

    # within ... -> upper_bound
    if "within" in text_low:
        within_phrase = text_low.split("within", 1)[1]
        m = RE_NUM_UNIT.search(within_phrase)
        if m:
            unit_c = _canon_unit(m.group("unit"))
            if "distance" in within_phrase:
                attr = add_attr("distance")
            elif "threshold" in within_phrase:
                attr = add_attr("avoidance threshold")
            else:
                attr = add_attr("value")
            add_pair(attr, "<=", float(m.group("num")), unit_c)

    # at least ... -> lower_bound for turn angle
    if "at least" in text_low:
        after = text_low.split("at least", 1)[1]
        m = RE_NUM_UNIT.search(after)
        if m:
            unit_c = _canon_unit(m.group("unit"))
            attr = add_attr("turn angle")
            add_pair(attr, ">=", float(m.group("num")), unit_c)

    # below ... -> strict_upper_bound
    if "below" in text_low:
        after = text_low.split("below", 1)[1]
        m = RE_NUM_UNIT.search(after)
        if m:
            unit_c = _canon_unit(m.group("unit"))
            attr = add_attr("battery voltage")
            add_pair(attr, "<", float(m.group("num")), unit_c)

    # every ... -> exact (type None)
    if "every" in text_low:
        after = text_low.split("every", 1)[1]
        m = RE_NUM_UNIT.search(after)
        if m:
            unit_c = _canon_unit(m.group("unit"))
            attr = add_attr("execution period")
            add_pair(attr, "=", float(m.group("num")), unit_c)

    return attributes_req, pairs


# =========================
# Wrapper + Relationships
# =========================
def extract_per_requirement(req):
    doc = nlp(req)
    systems = extract_systems(doc)
    activities = extract_activities(doc)
    attributes, pairs = extract_attributes_and_constraints(req)
    return systems, activities, attributes, pairs


def build_relationships(requirements):
    """
    Returns list of:
      (req_id, system, activity, attribute, constraint_dict)
    Printed as:
      (R1): (system, activity, attribute, constraint)
    """
    out = []
    for idx, req in enumerate(requirements, start=1):
        req_id = f"R{idx}"
        systems, activities, attributes, pairs = extract_per_requirement(req)

        if pairs:
            for sys in systems or [None]:
                for act in activities or [None]:
                    for attr, cons in pairs:
                        out.append((req_id, sys, act, attr, cons))
        else:
            for sys in systems or [None]:
                for act in activities or [None]:
                    out.append((req_id, sys, act, None, None))
    return out


# =========================
# MAIN
# =========================
if __name__ == "__main__":
    requirements = load_requirements("requirements_input.txt")
    relationships = build_relationships(requirements)

    # Optional: Print global lists (like before)
    systems_all, acts_all, attrs_all, cons_all = [], [], [], []
    s_seen, a_seen, at_seen = set(), set(), set()

    for req in requirements:
        s, a, at, pairs = extract_per_requirement(req)
        for x in s:
            _add_unique(x, systems_all, s_seen)
        for x in a:
            _add_unique(x, acts_all, a_seen)
        for x in at:
            _add_unique(x, attrs_all, at_seen)
        for _, c in pairs:
            if c not in cons_all:
                cons_all.append(c)

    print("SYSTEMS:", systems_all)
    print("ACTIVITIES:", acts_all)
    print("ATTRIBUTES:", attrs_all)
    print("CONSTRAINTS:", cons_all)

    print("EXTARACTED INFO AS REQUIREMENT:")
    for req_id, system, activity, attribute, constraint in relationships:
        print(f"({req_id}): ({system}, {activity}, {attribute}, {constraint})")

FileNotFoundError: Requirements file not found: requirements_input.txt. Make sure 'requirements_input.txt' exists in the repo next to main.py.